# Zinc Certificate Valuation: Three Bubble Definitions

All calculations and charts remain in the notebook; no new image or CSV is written. The approved physical benchmark is the volume-weighted basket of positive 99.97 and 99.98 zinc-ingot trades under eligible cash and cash-matching contracts.

In [1]:
from pathlib import Path
import numpy as np
import pandas as pd
import plotly.graph_objects as go
from plotly.subplots import make_subplots
from IPython.display import display

def find_project() -> Path:
    cwd = Path.cwd().resolve()
    for base in (cwd, *cwd.parents):
        if (base / 'data/processed/bubble/zinc_certificate_bubble.csv').exists():
            return base
        candidate = base / 'commodity/zinc'
        if (candidate / 'data/processed/bubble/zinc_certificate_bubble.csv').exists():
            return candidate
    raise FileNotFoundError('Could not locate commodity/zinc')

PROJECT = find_project()
P = PROJECT / 'data/processed'
benchmark = pd.read_csv(P / 'physical' / 'zinc_9798_cash_daily.csv')
physical_direct = pd.read_csv(P / 'bubble' / 'physical_vs_intrinsic_bubble.csv', parse_dates=['date'])
certificate_direct = pd.read_csv(P / 'bubble' / 'certificate_vs_intrinsic_bubble.csv', parse_dates=['date'])
primary = pd.read_csv(P / 'bubble' / 'zinc_certificate_bubble.csv', parse_dates=['date'])
regression = pd.read_csv(P / 'bubble' / 'intrinsic_regression.csv', parse_dates=['date'])
regression_metrics = pd.read_csv(P / 'bubble' / 'intrinsic_regression_metrics.csv')
print(PROJECT)

E:\Work\commodity\zinc


## Data summary and three bubble definitions

In [2]:
def describe_bubble(frame, column, label):
    values = frame[column]
    return {
        'definition': label, 'rows': len(frame),
        'first_date': frame['date'].min().date(), 'last_date': frame['date'].max().date(),
        'mean_pct': values.mean(), 'median_pct': values.median(),
        'min_pct': values.min(), 'max_pct': values.max(),
        'positive_days': values.gt(0).sum(), 'negative_days': values.lt(0).sum(),
    }

summary = pd.DataFrame([
    describe_bubble(physical_direct, 'physical_vs_intrinsic_bubble_pct', 'physical / intrinsic'),
    describe_bubble(certificate_direct, 'certificate_vs_intrinsic_bubble_pct', 'certificate / intrinsic'),
    describe_bubble(primary, 'certificate_bubble_pct', 'certificate / estimated physical (primary)'),
])
display(summary.round({'mean_pct': 2, 'median_pct': 2, 'min_pct': 2, 'max_pct': 2}))
display(pd.DataFrame({
    'metric': ['benchmark days','certificate days','exact anchors','interpolated primary days'],
    'value': [len(benchmark), len(certificate_direct), primary['physical_ratio_method'].eq('observed').sum(), primary['physical_ratio_method'].eq('linear_interpolation').sum()]
}))

,definition,rows,first_date,last_date,mean_pct,median_pct,min_pct,max_pct,positive_days,negative_days
0,physical / intrinsic,562,2009-08-16,2026-09-13,-13.35,-15.99,-36.02,22.28,75,487
1,certificate / intrinsic,208,2025-10-20,2026-09-16,-20.54,-19.49,-33.91,-10.54,0,208
2,certificate / estimated physical (primary),201,2025-10-26,2026-09-13,0.80,1.72,-19.58,16.97,124,77


,metric,value
0,benchmark days,562
1,certificate days,208
2,exact anchors,49
3,interpolated primary days,152


## Schematic diagram of three bubbles
Two straight graphs show the distance of each market with LME x USD. The third graph is the main measure: the distance between the certificate and the estimated internal physical price.

In [3]:
series = [
    (physical_direct, 'physical_vs_intrinsic_bubble_pct', 'Physical vs intrinsic', '#1976D2'),
    (certificate_direct, 'certificate_vs_intrinsic_bubble_pct', 'Certificate vs intrinsic', '#EF6C00'),
    (primary, 'certificate_bubble_pct', 'Certificate vs estimated physical: primary', '#00897B'),
]
fig = make_subplots(rows=3, cols=1, vertical_spacing=0.07,
    subplot_titles=[item[2] for item in series])
for row, (frame, column, title, color) in enumerate(series, start=1):
    fig.add_trace(go.Scatter(x=frame['date'], y=frame[column], name=title,
        line_color=color), row=row, col=1)
    fig.add_hline(y=0, line_color='#455A64', row=row, col=1)
    fig.update_yaxes(title_text='Bubble (%)', row=row, col=1)
fig.update_layout(height=950, template='plotly_white', hovermode='x unified')
fig.show()

## Intrinsic price components and physical benchmark

In [4]:
fig = make_subplots(rows=2, cols=1, vertical_spacing=0.1,
    subplot_titles=('99.97 + 99.98 physical vs LME-FX intrinsic',
                    'Certificate settlement vs LME-FX intrinsic'))
for row, frame, price_column in [(1, physical_direct, 'physical_price_irr_per_kg'),
                                  (2, certificate_direct, 'certificate_price_irr_per_kg')]:
    fig.add_trace(go.Scatter(x=frame['date'], y=frame[price_column],
        name='Observed price', line_color='#1976D2', showlegend=row == 1), row=row, col=1)
    fig.add_trace(go.Scatter(x=frame['date'], y=frame['intrinsic_price_irr_per_kg'],
        name='Intrinsic', line_color='#EF6C00', showlegend=row == 1), row=row, col=1)
    fig.update_yaxes(title_text='IRR/kg', row=row, col=1)
fig.update_layout(height=750, template='plotly_white', hovermode='x unified')
fig.show()
display(benchmark[['physical_trade_date_jalali','grades','grade_99_97_quantity',
    'grade_99_98_quantity','total_quantity','grade_99_97_weighted_price',
    'grade_99_98_weighted_price','physical_weighted_price']].tail(30))

,physical_trade_date_jalali,grades,grade_99_97_quantity,grade_99_98_quantity,total_quantity,grade_99_97_weighted_price,grade_99_98_weighted_price,physical_weighted_price
532,1404/12/23,99.97|99.98,25,100,125,4.381454e+06,4.426163e+06,4.417221e+06
533,1405/01/09,99.97,50,0,50,4.118773e+06,NaN,4.118773e+06
534,1405/01/16,99.97|99.98,50,50,100,4.118772e+06,4.160801e+06,4.139786e+06
535,1405/01/23,99.97|99.98,300,20,320,4.004098e+06,4.044957e+06,4.006652e+06
536,1405/01/30,99.97,225,0,225,4.091775e+06,NaN,4.091775e+06
537,1405/02/06,99.97|99.98,250,70,320,4.245136e+06,4.321747e+06,4.261895e+06
538,1405/02/27,99.97|99.98,75,60,135,5.098596e+06,5.699996e+06,5.365885e+06
539,1405/03/03,99.97|99.98,246,325,571,5.120234e+06,5.172481e+06,5.149972e+06
540,1405/03/10,99.97|99.98,85,25,110,5.094630e+06,5.146616e+06,5.106445e+06
541,1405/03/17,99.97|99.98,50,100,150,5.172795e+06,5.225578e+06,5.207984e+06


## anchors and interpolation of the main method

In [5]:
observed = primary['physical_ratio_method'].eq('observed')
fig = make_subplots(rows=2, cols=1, vertical_spacing=0.1,
    subplot_titles=('Observed anchors and interpolated ratio',
                    'Certificate settlement and estimated physical price'))
fig.add_trace(go.Scatter(x=primary['date'], y=primary['physical_ratio'],
    name='Physical / intrinsic ratio', line_color='#7B1FA2'), row=1, col=1)
fig.add_trace(go.Scatter(x=primary.loc[observed, 'date'],
    y=primary.loc[observed, 'physical_ratio'], mode='markers',
    name='Exact anchors', marker_color='#263238'), row=1, col=1)
for column, name, color in [
    ('certificate_price_irr_per_kg', 'Certificate settlement', '#1976D2'),
    ('estimated_physical_price_irr_per_kg', 'Estimated physical', '#EF6C00'),
]:
    fig.add_trace(go.Scatter(x=primary['date'], y=primary[column],
        name=name, line_color=color), row=2, col=1)
fig.add_trace(go.Scatter(x=primary.loc[observed, 'date'],
    y=primary.loc[observed, 'observed_physical_price_irr_per_kg'],
    mode='markers', name='Observed physical', marker_color='#263238'), row=2, col=1)
fig.update_yaxes(title_text='IRR/kg', row=2, col=1)
fig.update_layout(height=750, template='plotly_white', hovermode='x unified')
fig.show()
display(primary.loc[observed, ['date','certificate_price_irr_per_kg',
    'observed_physical_price_irr_per_kg','intrinsic_price_irr_per_kg',
    'certificate_bubble_pct']])

,date,certificate_price_irr_per_kg,observed_physical_price_irr_per_kg,intrinsic_price_irr_per_kg,certificate_bubble_pct
0,2025-10-26,3056519,2.835858e+06,3478706.00,7.781100
5,2025-11-02,3056501,2.785596e+06,3441035.00,9.725210
10,2025-11-09,2904594,2.793914e+06,3440415.00,3.961453
15,2025-11-16,3006513,2.810795e+06,3664316.25,6.963098
20,2025-11-23,3024196,2.761754e+06,3530708.00,9.502712
24,2025-11-30,3002288,2.775485e+06,3797771.25,8.171634
29,2025-12-07,3062958,2.867416e+06,3964993.20,6.819436
34,2025-12-14,3048531,2.948016e+06,4170995.10,3.409581
39,2025-12-21,3309535,3.002456e+06,3980272.00,10.227579
44,2025-12-28,3741989,3.259074e+06,4343878.10,14.817552


## Experimental method of regression and data age control
Regression is not a substitute for the original method. Model selection is done only with TimeSeriesSplit and the lowest out-of-sample RMSE.

In [6]:
display(regression_metrics.sort_values('timeseries_cv_rmse'))
age = certificate_direct[['date', 'lme_age_days', 'usd_age_days']].set_index('date')
fig = make_subplots(rows=2, cols=1, vertical_spacing=0.1,
    subplot_titles=('Primary vs experimental regression bubble',
                    'As-of source age on certificate dates'))
fig.add_trace(go.Scatter(x=primary['date'], y=primary['certificate_bubble_pct'],
    name='Primary interpolation', line_color='#1976D2'), row=1, col=1)
fig.add_trace(go.Scatter(x=regression['date'], y=regression['certificate_bubble_pct'],
    name='Experimental regression', line_color='#7B1FA2'), row=1, col=1)
fig.add_hline(y=0, line_color='#455A64', row=1, col=1)
for column, color in [('lme_age_days', '#EF6C00'), ('usd_age_days', '#00897B')]:
    fig.add_trace(go.Scatter(x=age.index, y=age[column], name=column,
        line_shape='hv', line_color=color), row=2, col=1)
fig.update_yaxes(title_text='Bubble (%)', row=1, col=1)
fig.update_yaxes(title_text='Days old', row=2, col=1)
fig.update_layout(height=750, template='plotly_white', hovermode='x unified')
fig.show()
display(age.describe().T)

,model,timeseries_cv_rmse,selected
0,proportional_no_intercept,366667.985768,1
1,polynomial_degree_2_ridge,440746.167689,0
2,linear_with_intercept,502222.039841,0


,count,mean,std,min,25%,50%,75%,max
lme_age_days,208.0,0.687500,0.934284,0.0,0.0,0.0,1.0,4.0
usd_age_days,208.0,0.024038,0.153538,0.0,0.0,0.0,0.0,1.0


## Standard market dashboard

This governed, read-only section uses the same presentation contract across commodity projects:
source coverage, physical and certificate activity, separate price panels, physical-goods
composition, and validated processed bubbles. It never writes raw data or constructs a missing
bubble. For Zinc, product comparability still follows the project-specific workflow.

In [7]:
from pathlib import Path
import sys

def locate_workspace(start=Path.cwd()):
    for candidate in [start, *start.parents]:
        if (candidate / "commodity" / "zinc").exists() and (candidate / "shared").exists():
            return candidate
    raise FileNotFoundError("Could not locate workspace root")

WORKSPACE_ROOT = locate_workspace()
if str(WORKSPACE_ROOT) not in sys.path:
    sys.path.insert(0, str(WORKSPACE_ROOT))

from shared.notebook_tools.commodity_dashboard import (
    goods_type_counts,
    load_markets,
    market_summary,
    plot_available_bubbles,
    plot_goods_type_counts,
    plot_market_prices,
    plot_trade_activity,
)

PROJECT_DIR = WORKSPACE_ROOT / "commodity" / "zinc"
physical_dashboard, certificate_dashboard = load_markets(
    PROJECT_DIR, "zinc", physical_filename=None
)
display(market_summary(physical_dashboard, certificate_dashboard))
plot_trade_activity(physical_dashboard, certificate_dashboard, "Zinc")
plot_market_prices(physical_dashboard, certificate_dashboard, "Zinc")
goods_count_table = plot_goods_type_counts(physical_dashboard, "Zinc", top_n=30)
display(goods_count_table)
bubble_series_plotted = plot_available_bubbles(PROJECT_DIR, "Zinc")

,source_records,positive_trade_records,trading_days,first_date,last_date
market,,,,,
physical,6398,3581,770,1387/07/28,1405/06/25
certificate,286,208,208,1404/07/28,1405/06/26


Distinct physical GoodsName labels: 12; records counted: 6,398


,goods_name,record_count,share_pct
0,شمش روی 99.97,1713,26.773992
1,شمش روی 99.98,1677,26.211316
2,خاک روی,1257,19.646765
3,شمش روی 99.96,799,12.488278
4,شمش روی99.95,517,8.080650
5,شمش روی 99.99,344,5.376680
6,شمش روی 99.94,45,0.703345
7,شمش روی 99.93,35,0.547046
8,شمش روی 99.92,7,0.109409
9,شمش روی 99.90,2,0.031260


## Historical bubble distribution

This section reads the standardized processed table and renders an interactive Plotly figure for
each bubble type. The panels show the observed distribution, empirical cumulative distribution
function F(x), and magnitude frequency P(|Bubble| >= |x|). Negative bubbles retain their sign in
the first two panels; the third panel measures magnitude only.

In [8]:
from pathlib import Path
import sys
import pandas as pd

def locate_distribution_workspace(start=Path.cwd()):
    for candidate in [start, *start.parents]:
        if (candidate / "shared").exists() and (candidate / "commodity/zinc").exists():
            return candidate
    raise FileNotFoundError("Could not locate workspace root")

distribution_workspace = locate_distribution_workspace()
if str(distribution_workspace) not in sys.path:
    sys.path.insert(0, str(distribution_workspace))

from shared.market_analysis.bubble_distribution import plot_distribution_plotly

distribution_project = distribution_workspace / "commodity/zinc"
distribution_files = list(
    (distribution_project / "data/processed/bubble").glob("*_bubble_distribution.csv")
)
if len(distribution_files) != 1:
    raise ValueError(f"Expected one named bubble distribution CSV, found {distribution_files}")
bubble_distribution = pd.read_csv(distribution_files[0], parse_dates=["observation_date"])
for series_id, series_distribution in bubble_distribution.groupby("series_id", sort=True):
    comparison = series_distribution["comparison"].iloc[0]
    figure = plot_distribution_plotly(series_distribution, comparison)
    figure.show()

display(bubble_distribution)

,commodity,bubble_type,point_type,series_id,comparison,source_file,observation_date,bubble_pct,empirical_cdf,percentile,abs_exceedance_probability,abs_exceedance_pct,observation_count
0,zinc,certificate_vs_intrinsic,exact_observed,certificate_intrinsic,Certificate vs intrinsic,certificate_vs_intrinsic_bubble.csv,2026-04-07,-33.908137,0.004808,0.480769,0.004808,0.480769,208
1,zinc,certificate_vs_intrinsic,exact_observed,certificate_intrinsic,Certificate vs intrinsic,certificate_vs_intrinsic_bubble.csv,2026-04-06,-32.808760,0.009615,0.961538,0.009615,0.961538,208
2,zinc,certificate_vs_intrinsic,exact_observed,certificate_intrinsic,Certificate vs intrinsic,certificate_vs_intrinsic_bubble.csv,2026-04-20,-31.079797,0.014423,1.442308,0.014423,1.442308,208
3,zinc,certificate_vs_intrinsic,exact_observed,certificate_intrinsic,Certificate vs intrinsic,certificate_vs_intrinsic_bubble.csv,2026-04-08,-30.696536,0.019231,1.923077,0.019231,1.923077,208
4,zinc,certificate_vs_intrinsic,exact_observed,certificate_intrinsic,Certificate vs intrinsic,certificate_vs_intrinsic_bubble.csv,2026-04-18,-30.356409,0.024038,2.403846,0.024038,2.403846,208
...,...,...,...,...,...,...,...,...,...,...,...,...,...
814,zinc,physical_vs_intrinsic,exact_observed,physical_intrinsic,Physical vs intrinsic,physical_vs_intrinsic_bubble.csv,2010-05-19,16.414129,0.992883,99.288256,0.478648,47.864769,562
815,zinc,physical_vs_intrinsic,exact_observed,physical_intrinsic,Physical vs intrinsic,physical_vs_intrinsic_bubble.csv,2013-10-09,16.752890,0.994662,99.466192,0.464413,46.441281,562
816,zinc,physical_vs_intrinsic,exact_observed,physical_intrinsic,Physical vs intrinsic,physical_vs_intrinsic_bubble.csv,2015-08-22,18.407564,0.996441,99.644128,0.362989,36.298932,562
817,zinc,physical_vs_intrinsic,exact_observed,physical_intrinsic,Physical vs intrinsic,physical_vs_intrinsic_bubble.csv,2016-01-10,19.279271,0.998221,99.822064,0.311388,31.138790,562
